In [2]:
# %%
import os
import sys
import cv2
import json
import torch
import numpy as np
import onnxruntime as ort
import matplotlib.pyplot as plt
import time
from tqdm import tqdm

# Configuration affichage
%matplotlib inline

# Ajout du dossier courant au path
sys.path.append(os.getcwd())

# Configuration globale des seuils (Alignés sur le papier officiel)
CONF_THRES = 0.001  # Très bas pour maximiser le Recall
IOU_THRES = 0.6     # Seuil NMS

In [3]:
# %%
class DatasetImage:
    def __init__(self, img_path, model_path=None):
        self.img_path = img_path
        self.model_path = model_path
        
        if not os.path.exists(self.img_path):
            raise FileNotFoundError(f"Image introuvable : {self.img_path}")
            
        self.img_bgr = cv2.imread(self.img_path)
        self.img_rgb = cv2.cvtColor(self.img_bgr, cv2.COLOR_BGR2RGB)
        self.h_orig, self.w_orig = self.img_rgb.shape[:2]
        
        self.da_path, self.ll_path, self.det_path = self._get_gt_paths()
        
        self.session = None
        
        # Caches Résultats
        self.pred_da = None
        self.pred_ll = None
        self.pred_boxes = None
        self.gt_da = None
        self.gt_ll = None
        self.gt_boxes = None
        self.inference_time = 0

    def _get_gt_paths(self):
        da_path = self.img_path.replace("images", "da_seg_annotations").replace(".jpg", ".png")
        ll_path = self.img_path.replace("images", "ll_seg_annotations").replace(".jpg", ".png")
        det_path = self.img_path.replace("images", "det_annotations").replace(".jpg", ".json")
        return da_path, ll_path, det_path

    def _resize_unscale(self, img, new_shape=(640, 640), color=114):
        shape = img.shape[:2]
        if isinstance(new_shape, int): new_shape = (new_shape, new_shape)

        r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
        new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
        dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]
        dw, dh = dw // 2, dh // 2

        if shape[::-1] != new_unpad:
            img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_AREA)

        canvas = np.full((new_shape[0], new_shape[1], 3), color, dtype=np.uint8)
        canvas[dh:dh + new_unpad[1], dw:dw + new_unpad[0], :] = img

        return canvas, r, dw, dh, new_unpad[0], new_unpad[1]

    def load_ground_truth(self):
        # 1. Masques
        if os.path.exists(self.da_path):
            self.gt_da = cv2.imread(self.da_path, cv2.IMREAD_GRAYSCALE)
            if self.gt_da.shape != (self.h_orig, self.w_orig):
                self.gt_da = cv2.resize(self.gt_da, (self.w_orig, self.h_orig), interpolation=cv2.INTER_NEAREST)
        
        if os.path.exists(self.ll_path):
            self.gt_ll = cv2.imread(self.ll_path, cv2.IMREAD_GRAYSCALE)
            if self.gt_ll.shape != (self.h_orig, self.w_orig):
                self.gt_ll = cv2.resize(self.gt_ll, (self.w_orig, self.h_orig), interpolation=cv2.INTER_NEAREST)

        # 2. Boîtes (JSON)
        self.gt_boxes = []
        if os.path.exists(self.det_path):
            try:
                with open(self.det_path, 'r') as f:
                    data = json.load(f)
                frames = data.get('frames', [])
                if frames:
                    objects = frames[0].get('objects', [])
                    for obj in objects:
                        if obj.get('category') in ['car', 'bus', 'truck', 'train']:
                            box2d = obj.get('box2d', {})
                            if box2d:
                                self.gt_boxes.append([
                                    float(box2d['x1']), float(box2d['y1']),
                                    float(box2d['x2']), float(box2d['y2'])
                                ])
            except Exception as e:
                pass
        self.gt_boxes = np.array(self.gt_boxes)

    def run_inference(self):
        if self.session is None:
            # Optimisation: ONNX Runtime provider
            providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
            self.session = ort.InferenceSession(self.model_path, providers=providers)

        # Preprocess
        img_resized, r, dw, dh, nw, nh = self._resize_unscale(self.img_rgb, (640, 640))
        img_input = img_resized.astype(np.float32) / 255.0
        img_input = (img_input - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
        img_input = img_input.transpose(2, 0, 1)[None].astype(np.float32)

        # Inference
        start_time = time.time()
        outputs = self.session.run(['det_out', 'drive_area_seg', 'lane_line_seg'], 
                                   {self.session.get_inputs()[0].name: img_input})
        self.inference_time = (time.time() - start_time) * 1000 # ms
        
        det_out, da_seg_out, ll_seg_out = outputs

        # --- SEGMENTATION ---
        da_mask = np.argmax(da_seg_out[:, :, dh:dh+nh, dw:dw+nw], axis=1)[0]
        ll_mask = np.argmax(ll_seg_out[:, :, dh:dh+nh, dw:dw+nw], axis=1)[0]
        
        self.pred_da = cv2.resize(da_mask.astype(np.uint8), (self.w_orig, self.h_orig), interpolation=cv2.INTER_NEAREST)
        self.pred_ll = cv2.resize(ll_mask.astype(np.uint8), (self.w_orig, self.h_orig), interpolation=cv2.INTER_NEAREST)

        # --- DETECTION ---
        self.pred_boxes = []
        try:
            from lib.core.general import non_max_suppression
            det_tensor = torch.from_numpy(det_out).float()
            
            # Application des seuils officiels
            nms_out = non_max_suppression(det_tensor, conf_thres=CONF_THRES, iou_thres=IOU_THRES)[0]
            
            if nms_out is not None and len(nms_out) > 0:
                nms_out = nms_out.cpu().numpy()
                nms_out[:, 0] -= dw
                nms_out[:, 1] -= dh
                nms_out[:, 2] -= dw
                nms_out[:, 3] -= dh
                nms_out[:, :4] /= r
                self.pred_boxes = nms_out
        except ImportError:
            pass

    def get_segmentation_stats(self, type='da'):
        """
        Retourne les stats pour le mIoU global.
        Applique une dilatation sur la GT des Lignes (LL) pour compenser l'épaisseur.
        """
        pred = self.pred_da if type == 'da' else self.pred_ll
        gt = self.gt_da if type == 'da' else self.gt_ll

        if pred is None or gt is None:
            return {'intersection': 0, 'union': 0}

        pred_bin = (pred > 0).astype(np.uint8)
        gt_bin = (gt > 0).astype(np.uint8)

        # --- DILATATION (Astuce pour les lignes) ---
        if type == 'll':
            kernel = np.ones((5, 5), np.uint8)
            gt_bin = cv2.dilate(gt_bin, kernel, iterations=1)

        intersection = np.logical_and(pred_bin, gt_bin).sum()
        union = np.logical_or(pred_bin, gt_bin).sum()
        
        return {'intersection': int(intersection), 'union': int(union)}

    def get_detection_stats(self, iou_threshold=0.5):
        """Retourne les détails pour le calcul mAP global."""
        if self.pred_boxes is None: self.pred_boxes = []
        if self.gt_boxes is None: self.gt_boxes = []

        gt_count = len(self.gt_boxes)
        pred_details = [] 

        if len(self.pred_boxes) == 0:
            return {'pred_details': [], 'gt_count': gt_count}

        preds = sorted(self.pred_boxes, key=lambda x: x[4], reverse=True)
        matched_gt = set()

        for pred in preds:
            bbox_pred = pred[:4]
            conf = pred[4]
            best_iou = 0
            best_gt_idx = -1

            for idx, gt in enumerate(self.gt_boxes):
                if idx in matched_gt: continue
                
                xA = max(bbox_pred[0], gt[0])
                yA = max(bbox_pred[1], gt[1])
                xB = min(bbox_pred[2], gt[2])
                yB = min(bbox_pred[3], gt[3])
                interArea = max(0, xB - xA) * max(0, yB - yA)
                boxAArea = (bbox_pred[2] - bbox_pred[0]) * (bbox_pred[3] - bbox_pred[1])
                boxBArea = (gt[2] - gt[0]) * (gt[3] - gt[1])
                iou = interArea / float(boxAArea + boxBArea - interArea + 1e-6)

                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx
            
            if best_iou >= iou_threshold and best_gt_idx != -1:
                pred_details.append([conf, True]) # TP
                matched_gt.add(best_gt_idx)
            else:
                pred_details.append([conf, False]) # FP
        
        return {'pred_details': pred_details, 'gt_count': gt_count}

    def get_speed_kpi(self):
        return self.inference_time
    

In [4]:
# %%
class DatasetFull:
    def __init__(self, img_dir, model_path, max_images=None):
        self.img_dir = img_dir
        self.model_path = model_path
        
        self.image_files = sorted([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
        if max_images:
            self.image_files = self.image_files[:max_images]
            
        print(f"📂 Dataset chargé : {len(self.image_files)} images.")

        # Stats Globales
        self.times = []
        
        # Segmentation Stats
        self.da_inter = 0
        self.da_union = 0
        self.ll_inter = 0
        self.ll_union = 0
        
        # Detection Stats
        self.det_preds = [] 
        self.det_gt_total = 0

    def process_all(self):
        print("🚀 Démarrage de l'évaluation complète...")
        
        for img_name in tqdm(self.image_files):
            img_path = os.path.join(self.img_dir, img_name)
            
            try:
                img_obj = DatasetImage(img_path, self.model_path)
                img_obj.load_ground_truth()
                img_obj.run_inference()
                
                # 1. Speed
                self.times.append(img_obj.get_speed_kpi())
                
                # 2. Seg Stats
                da = img_obj.get_segmentation_stats('da')
                self.da_inter += da['intersection']
                self.da_union += da['union']
                
                ll = img_obj.get_segmentation_stats('ll')
                self.ll_inter += ll['intersection']
                self.ll_union += ll['union']
                
                # 3. Det Stats
                det = img_obj.get_detection_stats(iou_threshold=0.5)
                self.det_preds.extend(det['pred_details'])
                self.det_gt_total += det['gt_count']
                
            except Exception as e:
                print(f"Erreur {img_name}: {e}")

    def compute_metrics(self):
        print("\n" + "="*50)
        print("📊 RÉSULTATS D'ÉVALUATION (YOLOP KPI)")
        print("="*50)
        
        # --- 1. Traffic Object Detection Result ---
        avg_ms = np.mean(self.times)
        fps = 1000 / avg_ms if avg_ms > 0 else 0
        
        preds = sorted(self.det_preds, key=lambda x: x[0], reverse=True)
        tp_list = [x[1] for x in preds]
        tp_cumsum = np.cumsum(tp_list)
        fp_cumsum = np.cumsum([not x for x in tp_list])
        
        recalls = tp_cumsum / (self.det_gt_total + 1e-16)
        precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-16)
        
        # mAP Calculation
        mrec = np.concatenate(([0.0], recalls, [1.0]))
        mpre = np.concatenate(([0.0], precisions, [0.0]))
        for i in range(mpre.size - 1, 0, -1):
            mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
        i = np.where(mrec[1:] != mrec[:-1])[0]
        ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
        
        final_recall = recalls[-1] if len(recalls) > 0 else 0

        print(f"Traffic Object Detection Result")
        print(f"Recall(%): {final_recall*100:.2f}")
        print(f"Speed(fps): {fps:.2f} ({avg_ms:.2f} ms)")
        print(f"mAP50(%): {ap*100:.2f}")
        print("-" * 50)

        # --- 2. Drivable Area Segmentation Result ---
        mIoU_da = (self.da_inter / self.da_union) * 100 if self.da_union > 0 else 0
        
        print(f"Drivable Area Segmentation Result")
        print(f"mIOU(%): {mIoU_da:.2f}")
        print(f"Speed(fps): {fps:.2f}")
        print("-" * 50)

        # --- 3. Lane Detection Result ---
        # Note: Pour les lignes, mIoU et IOU sont souvent confondus quand il y a 1 seule classe
        # Ici on calcule l'IoU de la classe positive (Ligne)
        iou_ll = (self.ll_inter / self.ll_union) * 100 if self.ll_union > 0 else 0
        # Pour le mIoU formel (Moyenne IoU Fond + IoU Ligne), on approxime souvent
        # que le fond est très bien classé (~99%).
        # Ici on affiche l'IoU de la ligne (le plus important).
        
        print(f"Lane Detection Result:")
        print(f"mIOU(%): {iou_ll:.2f} (Calculé sur la classe positive)")
        print(f"IOU(%): {iou_ll:.2f}") # Idem car 1 classe cible
        print("="*50)

In [5]:
# %%
if __name__ == "__main__":
    # Paramètres
    VAL_DIR = "dataset/images/val"
    ONNX_PATH = "weights/yolop-640-640.onnx"
    
    # Création et Lancement
    # Augmentez max_images (ex: 500 ou None pour tout) pour avoir des stats plus stables
    dataset_eval = DatasetFull(VAL_DIR, ONNX_PATH, max_images=200)
    
    # 1. Traitement
    dataset_eval.process_all()
    
    # 2. Affichage des KPI
    dataset_eval.compute_metrics()

📂 Dataset chargé : 200 images.
🚀 Démarrage de l'évaluation complète...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [01:25<00:00,  2.33it/s]


📊 RÉSULTATS D'ÉVALUATION (YOLOP KPI)
Traffic Object Detection Result
Recall(%): 90.07
Speed(fps): 2.89 (346.35 ms)
mAP50(%): 77.14
--------------------------------------------------
Drivable Area Segmentation Result
mIOU(%): 84.93
Speed(fps): 2.89
--------------------------------------------------
Lane Detection Result:
mIOU(%): 43.00 (Calculé sur la classe positive)
IOU(%): 43.00
